# Academic Journals - SPECTER2 Embedding-Analyse

Wandelt jedes Paper mit Titel und Abstract in einen Zahlen-Vektor um und misst, wie ähnlich (thematisch) sich Paper sind. Ergebnis: **topic_match**. Abschnitte der Reihe nach ausführen.

## 1 - Was macht SPECTER2?

SPECTER2 verwandelt Titel + Abstract eines Papers in einen Zahlen-Vektor. Ähnliche Themen ergeben ähnliche Vektoren. Damit können wir thematische Nähe zwischen Papern messen.

## 2 - Welche Autoren?

**v5:** Die Produktivitaetsschwelle ist auf 1 gesetzt, also faktisch aufgehoben.
Begruendung: Produktivitaet haengt mit Journaleintritten zusammen. Eine Schwelle
filtert damit auf einer Nachbehandlungsgroesse und erzeugt Selektionsverzerrung.
Zusaetzlich wirkte die Schwelle bis in die Journalprofile durch (siehe Abschnitt 8).

MIN_PAPERS = 3 reproduziert den alten Lauf (adjustierte RR 5,24) und ist ab v5
nur noch Sensitivitaetsvariante.

## 3 - Daten aus SharePoint

Wir verwenden denselben Release wie Q1/Q2 und Q3. Colab lädt die Rohdaten und die
Gelegenheitentabelle, prüft ihre SHA-256 und baut daraus seine lokale DuckDB.
Eine zusätzliche Datenbank in Google Drive brauchen wir nicht.

**Zugang zuerst klären:** Der normale rclone-Login erlaubt mehr als den Zugriff auf
unseren Gruppenordner. `SHARED_FOLDER` begrenzt nur die Dateipfade, nicht die Rechte des
Tokens. Dafür ein Projektkonto ohne andere Daten oder Freigaben verwenden. Alternativ
muss die Hochschule eine App mit [beschränkten Microsoft-Rechten](https://learn.microsoft.com/en-us/graph/permissions-selected-overview)
einrichten; diese Variante ist hier noch nicht getestet. Keine Zugangsdaten des normalen
Hochschulkontos in die gemeinsam bearbeitete Fassung übernehmen.

**Einmal einrichten:** Mit [rclone 1.75.1](https://rclone.org/downloads/) auf dem eigenen
Rechner `rclone --config academic-colab.conf config` ausführen und die OneDrive-Verbindung
`shared_data` anlegen. Eine eigene, geprüfte Notebook-Kopie verwenden. Den Inhalt der
Konfigurationsdatei unter Schlüsselsymbol → Secrets als `ACADEMIC_RCLONE_CONFIG` hinterlegen.
Secrets und Laufzeitdateien werden nicht automatisch mit dem Notebook geteilt. Aber Code,
den man selbst ausführt, kann auf freigegebene Secrets zugreifen. Die Konfiguration daher
nicht in Zellen, Git oder den Teamchat kopieren.

Erneuerte Tokens bleiben nur in der Colab-Laufzeit. Wenn die Anmeldung später abläuft
oder widerrufen wird, lokal `rclone --config academic-colab.conf config reconnect shared_data:`
ausführen und das Secret erneuern. Die Zugangsdaten werden nicht nach SharePoint geschrieben.

Wenn der Gruppenordner nur unter „Geteilt“ erscheint, zuerst in OneDrive eine Verknüpfung
unter „Meine Dateien“ hinzufügen. `SHARED_FOLDER` unten ist der Pfad relativ zu diesem Drive.
Die [rclone-Anleitung](https://rclone.org/onedrive/#can-not-access-shared-with-me-files)
beschreibt diesen Fall. Falls Microsoft Administratorfreigabe verlangt, ist die Verbindung
noch nicht nutzbar; der Browserzugriff allein reicht dafür nicht.

Danach die Zellen der Reihe nach ausführen, mit GPU (T4). Neue Ergebnisse landen zunächst
in Colab; die letzte Zelle überträgt sie nach `runs/<Lauf-ID>/C_topic_match/` im Gruppenordner.
Der offizielle Release bleibt erhalten. **Die Microsoft-Anmeldung und ein vollständiger
GPU-Lauf mit dieser Anbindung sind noch zu prüfen.**

In [ ]:
# Code laden und die persönliche OneDrive-Verbindung einrichten
from pathlib import Path
import atexit, configparser, os, shutil, subprocess, sys, tempfile
from google.colab import userdata

REPO = Path('/content/academic-journals')
CODE_REVISION = '1ce71dc91205f9bf4d1979c207e6dd58e33a32a2'  # geprüfter Loader und Manifest
SHARED_FOLDER = 'Applied AI Group B Data'  # ggf. Name der eigenen OneDrive-Verknüpfung
REMOTE = 'shared_data:' + SHARED_FOLDER.strip('/')

if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Mista-Kev/academic-journals.git',
                    str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REVISION], check=True)
if not (REPO / 'data-manifest.json').is_file():
    raise RuntimeError('Unter REPO fehlt das Projekt mit data-manifest.json.')
if subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() != CODE_REVISION:
    raise RuntimeError('Der vorhandene Checkout entspricht nicht dem geprüften Code-Stand. Eine neue Laufzeit verwenden.')
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
    raise RuntimeError('Der lokale Projektcode wurde geändert. Vor Freigabe des Secrets prüfen.')
sys.path.insert(0, str(REPO))
from project_data import load_manifest, matches, install, safe_path

# Dieselbe Version wie beim lokalen Transfertest; kein veraltetes Ubuntu-Paket.
import hashlib, io, platform, urllib.request, zipfile
RCLONE_VERSION = '1.75.1'
RCLONE = Path('/content/bin/rclone-1.75.1')
if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
    raise RuntimeError('Diese Installationszelle erwartet eine Linux-x86_64-Colab-Laufzeit.')
if not RCLONE.is_file():
    url = 'https://downloads.rclone.org/v1.75.1/rclone-v1.75.1-linux-amd64.zip'
    with urllib.request.urlopen(url, timeout=60) as response:
        archive = response.read()
    if hashlib.sha256(archive).hexdigest() != '982b5aa772841168f8e380f139e9e787b2a105403e32b94da8676a0e1c0a13ab':
        raise RuntimeError('Prüfsumme des rclone-Downloads stimmt nicht.')
    RCLONE.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
        binary = zipped.read('rclone-v1.75.1-linux-amd64/rclone')
    with tempfile.NamedTemporaryFile(dir=RCLONE.parent, delete=False) as out:
        temp_binary = Path(out.name)
        out.write(binary)
    temp_binary.chmod(0o755)
    temp_binary.replace(RCLONE)
version = subprocess.check_output([str(RCLONE), 'version'], text=True).splitlines()[0]
if version != 'rclone v' + RCLONE_VERSION:
    raise RuntimeError('Unerwartete rclone-Version; Installationsdatei prüfen.')
print(version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'], check=True)

_secret = userdata.get('ACADEMIC_RCLONE_CONFIG')
_config = configparser.ConfigParser(interpolation=None)
try:
    _config.read_string(_secret)
except configparser.Error:
    raise RuntimeError('Das Colab-Secret ist keine gültige rclone-Konfiguration.') from None
if _config.sections() != ['shared_data'] or _config['shared_data'].get('type') != 'onedrive':
    raise RuntimeError('Das Secret muss genau eine OneDrive-Verbindung [shared_data] enthalten.')
_auth_dir = tempfile.TemporaryDirectory(prefix='academic-auth-')
atexit.register(_auth_dir.cleanup)
_auth_file = Path(_auth_dir.name) / 'rclone.conf'
_auth_file.touch(mode=0o600)
_auth_file.write_text(_secret)
del _secret, _config

def transfer(source, destination):
    # Keine Zugangsdaten oder Serverantworten in gespeicherten Notebook-Ausgaben.
    result = subprocess.run([str(RCLONE), '--config', str(_auth_file), 'copyto',
                             str(source), str(destination), '--ignore-existing'],
                            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    if result.returncode:
        error = result.stderr.lower()
        # Nur feste Hinweise ausgeben: auch eine letzte stderr-Zeile kann Tokens enthalten.
        if any(x in error for x in (b'401', b'unauthorized', b'invalid_grant', b'token expired')):
            hint = 'Microsoft-Anmeldung erneuern und Colab-Secret aktualisieren.'
        elif any(x in error for x in (b'403', b'accessdenied', b'access denied', b'forbidden')):
            hint = 'Dem Konto fehlt der Zugriff oder Microsoft verweigert die App.'
        elif result.returncode in (3, 4) or any(x in error for x in (b'404', b'directory not found', b'object not found')):
            hint = 'SHARED_FOLDER, Freigabe-Verknüpfung und Release-Pfad prüfen.'
        else:
            hint = 'Verbindung und Zugriff prüfen; lokale Ergebnisse bleiben erhalten.'
        raise RuntimeError(f'Dateitransfer fehlgeschlagen (rclone {result.returncode}). ' + hint)

manifest = load_manifest(REPO)
print('Datenstand:', manifest['release'])

In [ ]:
# Nur die beiden benötigten Eingaben laden; der Repo-Manifest legt den Stand fest.
RAW_PATH = 'A_data_and_rules/data/openalex_ai_raw_v1_0.jsonl'
EVENT_PATH = 'B_opportunities_and_analysis/data/event_table_python_v0_oppA.csv'
release_remote = REMOTE + '/' + manifest['release'] + '-' + manifest['layout']
with tempfile.TemporaryDirectory() as staging:
    shared_manifest = Path(staging) / 'data-manifest.json'
    transfer(release_remote + '/data-manifest.json', shared_manifest)
    if shared_manifest.read_bytes() != (REPO / 'data-manifest.json').read_bytes():
        raise RuntimeError('SharePoint-Release und Repository haben unterschiedliche Manifeste.')
for relative in (RAW_PATH, EVENT_PATH):
    item = next(x for x in manifest['files'] if x['path'] == relative)
    target = safe_path(REPO, relative)
    if target.exists():
        if not matches(target, item):
            raise RuntimeError('Lokale Datei weicht vom Release ab: ' + relative)
    else:
        with tempfile.TemporaryDirectory() as staging:
            downloaded = Path(staging) / 'input'
            transfer(release_remote + '/' + relative, downloaded)
            with downloaded.open('rb') as stream:
                install(stream, target, item)
    print('Geladen und geprüft:', relative)

# Die JSONL enthält dieselben OpenAlex-Felder, die die folgenden SQL-Abfragen nutzen.
# Explizite Typen vermeiden eine zufällige Schema-Auswahl aus einer JSON-Stichprobe.
import duckdb
LOCAL_DB = '/content/academic_journals.duckdb'
with duckdb.connect(LOCAL_DB) as db:
    db.execute("""
        CREATE OR REPLACE TABLE openalex_ai_raw_v1_0 AS
        SELECT * FROM read_json(?, format='newline_delimited', columns={
            'id': 'VARCHAR', 'title': 'VARCHAR', 'publication_year': 'INTEGER',
            'publication_date': 'DATE',
            'abstract_inverted_index': 'MAP(VARCHAR, INTEGER[])',
            'authorships': 'STRUCT(author STRUCT(id VARCHAR, display_name VARCHAR))[]',
            'primary_location': 'STRUCT(source STRUCT(id VARCHAR, display_name VARCHAR,
                                  host_organization VARCHAR, is_in_doaj BOOLEAN))'
        })
    """, [str(REPO / RAW_PATH)])
    paper_count = db.execute('SELECT count(*) FROM openalex_ai_raw_v1_0').fetchone()[0]
    print('Paper in DuckDB:', paper_count)
    assert paper_count == 27400, 'Der geprüfte Korpus muss 27.400 Paper enthalten.'

from datetime import datetime, timezone
from uuid import uuid4
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
RUN_ROOT = Path('/content/topic-runs') / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
LOCAL_CSV = RUN_ROOT / f'C_topic_match/results/results_q1_topic_match_run_{RUN_ID}.csv'
LOCAL_KEYS = RUN_ROOT / f'C_topic_match/results/keys_author_paper_run_{RUN_ID}.csv'
EVENT_IN = REPO / EVENT_PATH
EVENT_OUT = RUN_ROOT / f'C_topic_match/data/event_table_topicmatch_run_{RUN_ID}.csv'
for output in (LOCAL_CSV, LOCAL_KEYS, EVENT_OUT):
    output.parent.mkdir(parents=True, exist_ok=True)
print('Neue Ergebnisse:', RUN_ROOT)

## 4 - Setup

Pakete installieren und pruefen, ob eine GPU aktiv ist.

In [ ]:
# Pakete installieren + GPU pruefen
# v5: Installation NICHT stillschweigend uebergehen. Im v4-Lauf war
# "pip install" mit "Operation cancelled by user" abgebrochen, wodurch eine
# vorinstallierte adapters-Version verwendet wurde - vermutlich die Ursache
# der Adapter-Warnung. Die Versionen werden deshalb ausgegeben.
!pip install -q duckdb transformers adapters

import importlib.metadata as _md
for _p in ("duckdb", "transformers", "adapters", "torch", "pandas", "numpy"):
    try:
        print(f"{_p:15s} {_md.version(_p)}")
    except Exception:
        print(f"{_p:15s} NICHT INSTALLIERT")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("\nDevice:", DEVICE)
assert DEVICE == "cuda", "Ohne GPU dauert der Lauf Stunden - Runtime auf T4 umstellen."


## 5 - Autoren und Paper laden

Alle produktiven Autoren auswählen und ihre Paper aus der Datenbank ziehen.

In [ ]:
# Autoren auswaehlen
import duckdb, numpy as np

TBL         = "openalex_ai_raw_v1_0"   # Rohdatenbasis

# ---------------------------------------------------------------------------
# MIN_PAPERS steuert AUSSCHLIESSLICH die Autorenprofile.
#   1 = schwellenfrei (Hauptlauf ab v5)
#   3 = alter Lauf, nur noch Sensitivitaetsvariante
# Die Journalprofile sind ab v5 von dieser Zahl entkoppelt (Abschnitt 5).
# ---------------------------------------------------------------------------
MIN_PAPERS  = 1

con = duckdb.connect(LOCAL_DB, read_only=True)

# Autor -> Paper: nur Paper mit Titel UND Abstract (= einbettbar)
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW author_paper AS
    WITH exploded AS (
        SELECT id AS work_id,
               unnest(authorships) AS a
        FROM {TBL}
        WHERE title IS NOT NULL AND abstract_inverted_index IS NOT NULL
    )
    SELECT a.author.id AS author_id, a.author.display_name AS author_name, work_id
    FROM exploded
    WHERE a.author.id IS NOT NULL
""")

prod = con.execute(f"""
    SELECT author_id, author_name, COUNT(DISTINCT work_id) AS n_paper
    FROM author_paper
    GROUP BY author_id, author_name
    HAVING COUNT(DISTINCT work_id) >= {MIN_PAPERS}
    ORDER BY n_paper DESC
""").fetchall()

chosen_ids = [p[0] for p in prod]
print(f"{len(chosen_ids)} Autoren mit >= {MIN_PAPERS} einbettbaren Papern.")

# chosen_ids als Tabelle ablegen - vermeidet spaeter IN-Listen mit
# zehntausenden Platzhaltern (bei MIN_PAPERS = 1 sind das ~82.000 IDs)
con.execute("CREATE OR REPLACE TEMP TABLE chosen(author_id VARCHAR)")
con.executemany("INSERT INTO chosen VALUES (?)", [(a,) for a in chosen_ids])
print("Temp-Tabelle 'chosen' angelegt.")

### v5: Warum hier alle Paper geladen werden

Bis v4 wurden nur die Paper der ausgewaehlten Autoren geladen. Die Kette war:

`MIN_PAPERS` -> `chosen_ids` -> `work_ids` -> `meta` -> `journal_pairs`

Damit hing das **Journalprofil** an der Autorenauswahl. Ein Journal mit 400
Papern in einem Jahr bekam ein Profil aus den 60 Papern, die von einem
ausgewaehlten Autor stammten. Das ist derselbe Fehlertyp wie beim frueheren
`n_journal_papers`-Bug, nur eine Ebene tiefer.

Ab v5 ist der Ladeschritt entkoppelt. Das **Autorenprofil** bleibt ueber
`author_works` auf `chosen_ids` beschraenkt - dort ist die Auswahl richtig.

Die Pruefung in Abschnitt 8 vergleicht die Journalzaehlungen gegen eine
unabhaengige Abfrage in der Datenbank und bricht ab, wenn sie abweichen.

In [ ]:
# Paper laden - ALLE einbettbaren Paper des Korpus
#
# v5-AENDERUNG. Vorher wurde hier gegen work_ids der ausgewaehlten Autoren
# gejoint. Damit hing der Embedding-Korpus an MIN_PAPERS - und ueber meta
# auch das Journalprofil (Abschnitt 8). Ein Journalprofil muss eine
# Eigenschaft des Journals sein, nicht der Autorenauswahl.
#
# Die Autorenprofile bleiben ueber author_works auf chosen_ids beschraenkt.
# Dort ist die Auswahl richtig und beabsichtigt.

rows = con.execute(f"""
    SELECT
        w.id                                          AS work_id,
        w.title,
        w.abstract_inverted_index                     AS abs_idx,
        w.publication_year                            AS year,
        COALESCE(strftime(w.publication_date, '%Y-%m-%d'),
                 CAST(w.publication_year AS VARCHAR) || '-01-01') AS pub_date,
        w.primary_location.source.id                  AS journal_id,
        w.primary_location.source.display_name        AS journal_name,
        w.primary_location.source.host_organization   AS publisher_id,
        w.primary_location.source.is_in_doaj          AS in_doaj
    FROM {TBL} w
    WHERE w.title IS NOT NULL AND w.abstract_inverted_index IS NOT NULL
""").fetchall()

print(f"{len(rows)} einbettbare Paper im Korpus (unabhaengig von MIN_PAPERS).")

## 6 - Abstract wiederherstellen

OpenAlex speichert Abstracts als Wort-Positionen. Wir setzen daraus wieder normalen Text zusammen.

In [ ]:
# Abstract aus Wort-"Fetzen" zusammensetzen
def reconstruct_abstract(inv):
    if not inv:
        return ""
    pairs = []
    for word, positions in inv.items():
        for p in positions:
            pairs.append((p, word))
    pairs.sort(key=lambda x: x[0])
    return " ".join(w for _, w in pairs)

# v5.1: Paper, deren Abstract-Index zwar vorhanden ist, aber zu leerem Text
# zerfaellt, fallen hier heraus. Sie werden protokolliert, damit die
# Journalprofil-Pruefung in Abschnitt 14 sie herausrechnen kann. Ohne diese
# Buchhaltung vergleicht die Pruefung zwei verschieden gefilterte Mengen.
from collections import defaultdict as _dd
dropped_empty_abstract = _dd(lambda: _dd(int))   # journal_id -> jahr -> anzahl
n_dropped = 0

titles, abstracts, meta = [], [], []
for work_id, title, abs_idx, year, pub_date, journal_id, journal_name, publisher_id, in_doaj in rows:
    abstract = reconstruct_abstract(abs_idx)
    if not abstract.strip():
        n_dropped += 1
        if journal_id and year is not None:
            dropped_empty_abstract[journal_id][int(year)] += 1
        continue
    titles.append(title)
    abstracts.append(abstract)
    meta.append({"work_id": work_id, "year": year, "date": pub_date,
                 "journal_id": journal_id, "journal_name": journal_name,
                 "publisher_id": publisher_id, "in_doaj": in_doaj})

print(f"{len(titles)} Paper mit nutzbarem Abstract.")
print(f"{n_dropped} Paper verworfen: Abstract-Index vorhanden, Text leer.")
assert len(titles) + n_dropped == len(rows), "Buchhaltung stimmt nicht"
print("\nBeispiel-Titel   :", titles[0][:90])
print("Beispiel-Abstract:", abstracts[0][:200], "...")

# Vor dem GPU-Lauf: dieselbe nutzbare Paperbasis wie im geprüften Release.
assert len(titles) == 26962, "Erwartet: 26.962 Paper mit nutzbarem Abstract. Datenbasis prüfen."


## 7 - Paper einbetten

SPECTER2 lädt und macht aus jedem Paper einen Zahlen-Vektor

In [ ]:
# SPECTER2 laden und alle Paper einbetten
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

tok   = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter("allenai/specter2", source="hf", set_active=True)

# v5: Adapter-Status pruefen, NICHT annehmen.
# Der produktive v4-Lauf meldete trotz set_active=True eine Warnung, dass
# fuer den Forward Pass kein Adapter aktiv ist - im selben Lauf war die
# pip-Installation abgebrochen. Ohne aktiven Proximity-Adapter rechnet das
# Basismodell, was fuer Aehnlichkeitsaufgaben nicht die Referenzkonfiguration
# ist. Hier bricht der Lauf ab, statt still das Falsche zu rechnen.
_active = model.active_adapters
print("aktive Adapter:", _active)
assert _active is not None, (
    "Kein Adapter aktiv. Bibliothek 'adapters' pruefen "
    "(pip install -q adapters muss vollstaendig durchlaufen)."
)

model.to(DEVICE).eval()

def embed(titles, abstracts, batch_size=32):
    texts = [(t or "") + tok.sep_token + (a or "") for t, a in zip(titles, abstracts)]
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tok(batch, padding=True, truncation=True,
                  return_tensors="pt", max_length=512).to(DEVICE)
        with torch.no_grad():
            o = model(**inp)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())  # CLS
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    return np.vstack(out)

X = embed(titles, abstracts)
X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)   # L2-normalisieren
print(f"\nEmbeddings: {X.shape}   (Paper x 768)")

## 8 - Ähnlichkeit Kernel - Matrix

Der RBF-Kernel misst die Ähnlichkeit zwischen zwei Vektoren. Wir rechnen nur die nötigen Vergleiche, nie die riesige Gesamtmatrix - das spart Speicher. Müssen wir aber noch ausführen, warum wir so vorgehen

In [ ]:
# Kernel-Ähnlichkeit, speicherschonend
_rng = np.random.default_rng(0)
_m = min(3000, len(X))
_idx = _rng.choice(len(X), _m, replace=False)
_Xs = X[_idx]
_s = np.sum(_Xs**2, axis=1)
_D2 = np.maximum(_s[:, None] + _s[None, :] - 2 * _Xs @ _Xs.T, 0.0)
sig2 = float(np.median(_D2[np.triu_indices(_m, k=1)]))
print(f"sigma^2 (Median-Heuristik, Stichprobe {_m}) = {sig2:.4f}")

def kernel_group_mean(idxs):
    """Mittlere paarweise RBF-Ähnlichkeit innerhalb einer kleinen Gruppe."""
    if len(idxs) < 2:
        return None
    sub = X[idxs]
    s = np.sum(sub**2, axis=1)
    D2 = np.maximum(s[:, None] + s[None, :] - 2 * sub @ sub.T, 0.0)
    K = np.exp(-0.5 * D2 / sig2)
    iu = np.triu_indices(len(idxs), k=1)
    return float(np.mean(K[iu]))

def sim_to_all(i):
    """RBF-Ähnlichkeit eines Ankers zu ALLEN Papern (ein Vektor, keine Matrix)."""
    d = X - X[i]
    D2 = np.sum(d**2, axis=1)
    return np.exp(-0.5 * D2 / sig2)

# Zentrale Zuordnungen (von hier an notwendig)
row_of        = {m["work_id"]: i for i, m in enumerate(meta)}
journal_id_of = {m["work_id"]: m["journal_id"] for m in meta}
emb_ids       = set(row_of)

## 9 - Testblock

Prüfen, ob ähnliche Paper wirklich thematisch passen, und die "mittlere Ähnlichkeit" von Papern desselben Autors im selben Journal messen.

In [ ]:
# Ähnlichste Paper als Anker ermitteln --> Test
def most_similar(i, k=3):
    sims = sim_to_all(i); sims[i] = -1
    for j in np.argsort(sims)[::-1][:k]:
        print(f"   {sims[j]:.3f}  {titles[j][:75]}")

for a in [0, 1, 2]:
    print(f"\nAnker: {titles[a][:75]}")
    most_similar(a)

In [ ]:
# Q1: mittlere Themen-Aehnlichkeit --> selbes Journal
from collections import defaultdict

# v5: eine Abfrage statt einer pro Autor. Bei MIN_PAPERS = 1 waeren das
# ~82.000 Einzelabfragen gewesen.
author_works = defaultdict(list)
for aid, w in con.execute("""
    SELECT DISTINCT ap.author_id, ap.work_id
    FROM author_paper ap
    JOIN chosen c ON c.author_id = ap.author_id
""").fetchall():
    if w in emb_ids:
        author_works[aid].append(w)

print(f"{len(author_works)} Autoren mit mindestens einem eingebetteten Paper.")

def author_intra_journal(work_ids):
    by_j = defaultdict(list)
    for w in work_ids:
        by_j[journal_id_of[w]].append(row_of[w])
    vals = [kernel_group_mean(idxs) for idxs in by_j.values() if len(idxs) >= 2]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

scores = [(a, author_intra_journal(ws)) for a, ws in author_works.items()]
scores = [(a, s) for a, s in scores if s is not None]
if scores:
    arr = np.array([s for _, s in scores])
    print(f"Autoren mit >=2 Papern im selben Journal: {len(scores)}")
    print(f"Mittlere Intra-Journal-Aehnlichkeit: {arr.mean():.3f} (SD {arr.std():.3f})")
else:
    print("Keine Autoren mit >=2 Papern im selben Journal.")

## 10 - Was bedeuten die Zahlen?

topic_match liegt zwischen 0 und 1: hoch = thematisch ähnlich, niedrig = fremd. Weil der Datensatz nur KI-Paper enthält, sind die Werte generell hoch - wir lesen **Unterschiede**, nicht absolute Ausprägungen.

## 11 - Anschluss an die anderen Layer

Alle drei Layer verbinden sich über die **work_id**. Unser topic_match ist die Themen-Variable, die der Probabilistic-Layer für die Q3-Frage nutzt.

## 12 - Schlüssel-Tabelle

Alle Autor-Paper-Zeilen mit IDs - die Basis, die der Logic-Layer braucht.

In [ ]:
# Schlüssel-Tabelle (Autor und Paper Zeiln)
keys = con.execute(f"""
    WITH exploded AS (
        SELECT id AS work_id,
               publication_year                            AS year,
               primary_location.source.id                  AS journal_id,
               primary_location.source.display_name        AS journal_name,
               primary_location.source.host_organization   AS publisher_id,
               primary_location.source.is_in_doaj          AS in_doaj,
               unnest(authorships)                          AS a
        FROM {TBL}
    )
    SELECT work_id, year, journal_id, journal_name, publisher_id, in_doaj,
           a.author.id           AS author_id,
           a.author.display_name AS author_name
    FROM exploded
    WHERE a.author.id IS NOT NULL
""").fetchall()

print(f"{len(keys)} Autor-Paper-Zeilen.")
print("Spalten: work_id, year, journal_id, journal_name, publisher_id, in_doaj, author_id, author_name")
print("Beispiel:", keys[0])

## 13 - Ergebnisse exportieren

Die beiden CSVs werden im lokalen Laufordner gespeichert. Nach der vollständigen
Berechnung überträgt die letzte Zelle alle drei Ergebnisse in den gemeinsamen SharePoint-Ordner.

In [ ]:
# Export: Q1-Ergebnis als CSV ausgeben
import csv

rows_out = []
for aid, ws in author_works.items():
    by_j = defaultdict(list)
    for w in ws:
        by_j[journal_id_of[w]].append(w)
    for jid, wids in by_j.items():
        if len(wids) >= 2:
            val = kernel_group_mean([row_of[w] for w in wids])
            rows_out.append({
                "author_id": aid,
                "journal_id": jid,
                "n_papers": len(wids),
                "topic_match_intra": round(val, 4),
                "work_ids": ";".join(wids),
            })

with open(LOCAL_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=["author_id","journal_id","n_papers",
                                       "topic_match_intra","work_ids"])
    wr.writeheader(); wr.writerows(rows_out)

print(f"{len(rows_out)} Autor-Journal-Zeilen exportiert (voller Lauf).")
if rows_out:
    print("Beispiel:", rows_out[0])

In [ ]:
# Export: Das ist die Schlüssel-Tabelle, ebenfalls als CSV
import csv

with open(LOCAL_KEYS, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["work_id","year","journal_id","journal_name",
                 "publisher_id","in_doaj","author_id","author_name"])
    wr.writerows(keys)

print(f"{len(keys)} Zeilen exportiert nach {LOCAL_KEYS.name}")

## 14 - topic_match für die Event-Tabelle füllen

Statt eigene Gelegenheiten zu bauen, lesen wir die **Event-Tabelle des Logic-Layers**
ein und füllen nur die leere Spalte topic_match. Pro Zeile (Autor, Journal, Jahr):
Ähnlichkeit zwischen **Autor-Profil** und **Journal-Profil**, beide nur aus Papern
**vor** dem Jahr. Auf **Jahresebene**, wie der Logic-Layer rechnet.

Zeilen ohne Vergangenheits-Profil - oder deren Autor nicht eingebettet ist (nur
produktive Autoren haben Embeddings) - bleiben **leer** (nicht 0). Wichtig für die Weiterverarbeitung!!!

In [ ]:
# Geprüfte Gelegenheitentabelle aus B einlesen; Ganzzahlspalten erhalten
import pandas as pd
import numpy as np


COL_AUTHOR  = "author_id"
COL_JOURNAL = "journal_id"
COL_YEAR    = "t"             # das Jahr t der Zeile
COL_TOPIC   = "topic_match"   # wird von uns befuellt (SPECTER2)

# Spalten, die echte Kommazahlen sind und NICHT zu Int64 werden duerfen
FLOAT_COLS = {COL_TOPIC}

ev = pd.read_csv(EVENT_IN)
print(f"{len(ev):,} Zeilen. Spalten: {list(ev.columns)}")

# --- Ganzzahl-Typen wiederherstellen -------------------------------------
# pandas macht aus jeder Integer-Spalte MIT Leerwerten automatisch float64
# (2020 -> 2020.0). Beim Zurueckschreiben landet das im CSV. Wir drehen das
# zurueck: jede float-Spalte, deren Werte alle ganzzahlig sind, wird Int64.
restored = []
for c in ev.columns:
    if c in FLOAT_COLS:
        continue
    if pd.api.types.is_float_dtype(ev[c]):
        v = ev[c].dropna()
        if len(v) and np.all(np.mod(v.values, 1) == 0):
            ev[c] = ev[c].astype("Int64")
            restored.append(c)

print("Als Int64 wiederhergestellt:", restored if restored else "(keine)")
print("dtypes:\n", ev.dtypes.to_string())

### Konsistenz-Check: passen Event-Tabelle und Embeddings zusammen?

Bevor wir rechnen: stammen beide aus derselben Datenbasis? Wir vergleichen nicht
Datei-Hashes (CSV vs. JSONL haben immer verschiedene Hashes), sondern den **Inhalt** -
wie viele unserer eingebetteten Autoren in der Event-Tabelle vorkommen und umgekehrt. Das ist ein Sicherheitscheck --> da wir die Datenbasis nie über das GitHub sharen

In [ ]:
# Überlappung der IDs zwischen Embeddings und Event-Tabelle prüfen --> Ich war mir nicht sicher, ob die Duck-DB-Basis zur Datenbasis von Kevin passt
emb_authors = set(author_works)                      # --> alle produktiven Autoren
emb_journals = {m["journal_id"] for m in meta}
ev_authors = set(ev[COL_AUTHOR].unique())
ev_journals = set(ev[COL_JOURNAL].unique())

# Wie viele unserer produktiven Autoren tauchen in der Event-Tabelle auf?
a_hit = len(emb_authors & ev_authors)
j_hit = len(emb_journals & ev_journals)

print("=== Autoren ===")
print(f"  eingebettet (produktiv)      : {len(emb_authors):,}")
print(f"  davon in der Event-Tabelle   : {a_hit:,} ({a_hit/max(len(emb_authors),1):.1%})")
print("=== Journale ===")
print(f"  in unseren Embeddings        : {len(emb_journals):,}")
print(f"  davon in der Event-Tabelle   : {j_hit:,} ({j_hit/max(len(emb_journals),1):.1%})")

# ID-Format-Check: sehen die IDs ueberhaupt gleich aus?
ex_emb = next(iter(emb_authors))
ex_ev  = next(iter(ev_authors))
print("\n=== ID-Format ===")
print(f"  Embedding-Autor-ID : {ex_emb}")
print(f"  Event-Autor-ID     : {ex_ev}")

if a_hit / max(len(emb_authors), 1) >= 0.9:
    print("\nOK: hohe Überlappung -> gleiche Basis, DuckDB muss NICHT neu gebaut werden.")
elif a_hit == 0:
    print("\nACHTUNG: keine Überlappung. Wahrscheinlich unterschiedliches ID-Format")
    print("  (z.B. volle URL vs. Kurzform) ODER verschiedene Datenversionen.")
    print("  Erst klaeren, bevor gefüllt wird - sonst bleibt alles leer.")
else:
    print("\nTeilweise Überlappung - vor dem Füllen pruefen, woran die Lücke liegt.")

In [ ]:
def short_id(x):
    return str(x).rsplit("/", 1)[-1]   # "https://openalex.org/A123" -> "A123"

# author_works, meta und row_of auf Kurz-IDs umstellen
author_works = {short_id(a): ws for a, ws in author_works.items()}
for m in meta:
    m["author_id"]  = short_id(m.get("author_id", ""))
    m["journal_id"] = short_id(m["journal_id"])
journal_id_of = {w: short_id(j) for w, j in journal_id_of.items()}

### Diagnosespalten und Statuscodes

Zusaetzlich zu `topic_match` schreibt der Lauf vier Spalten, damit der
Logic- und der Probabilistic-Layer leere Werte unterscheiden koennen:

| Spalte | Bedeutung |
| --- | --- |
| `n_profile_papers` | Paper des Autors **strikt vor t** (im Embedding-Set) |
| `n_journal_papers` | Paper des Journals **strikt vor t** (im Embedding-Set) |
| `profile_cutoff` | tatsaechliches letztes Jahr im Autor-Profil (nicht t-1) |
| `tm_status` | Grund fuer einen leeren Wert |

**Statuscodes**

| Code | Bedeutung |
| --- | --- |
| `ok` | Autor- und Journal-Profil vorhanden, Wert gerechnet |
| `no_author_history` | Autor hat vor t kein eingebettetes Paper |
| `no_journal_history` | **Journal** hat vor t kein eingebettetes Paper |
| `no_author_and_journal_history` | beides fehlt |
| `below_threshold_no_abstract` | Autor hat >= MIN_PAPERS Werke, aber < MIN_PAPERS mit Abstract |
| `below_threshold_unproductive` | Autor hat < MIN_PAPERS Werke insgesamt |
| `author_not_in_db` | Autor-ID nicht in der Rohdatenbasis |

`no_journal_history` ist der bisher unsichtbare Fall: Journal-Profile entstehen
nur aus Papern der ausgewaehlten produktiven Autoren. Hat ein Journal vor t kein
solches Paper, bleibt die Zeile leer - auch wenn der Autor sauber ueber der
Schwelle liegt und Vorhistorie hat.


In [ ]:
# Jahres-Profile ermitteln, topic_match fuellen + Diagnosespalten
# v4: n_journal_papers wird fuer ALLE Zeilen gerechnet (Eigenschaft des
#     Journal-Jahres, nicht der Zeile). n_profile_papers und profile_cutoff
#     bleiben leer, wo kein Profil existiert - eine 0 waere eine Falschaussage.
# v5: journal_pairs entsteht aus dem vollstaendigen Korpus (Abschnitt 5) und
#     ist damit von MIN_PAPERS entkoppelt. Die Pruefung unten belegt das
#     gegen eine unabhaengige Zaehlung in der Datenbank.
import numpy as np, bisect
from collections import defaultdict

year_of = {m["work_id"]: int(m["year"]) for m in meta if m["year"] is not None}

def build_year_centroids(pairs):
    out = {}
    for k, items in pairs.items():
        items = sorted(items)                       # nach Jahr
        ys = [y for y, _ in items]
        cs = np.cumsum(X[[r for _, r in items]], axis=0)
        out[k] = (ys, cs)
    return out

author_pairs = defaultdict(list)
for aid, ws in author_works.items():
    for w in ws:
        if w in year_of and w in row_of:
            author_pairs[aid].append((year_of[w], row_of[w]))

# Journalprofile: aus meta, das ab v5 den vollen Korpus enthaelt.
# Kein Bezug auf chosen_ids oder author_works - das ist der Kern der Aenderung.
journal_pairs = defaultdict(list)
for i, m in enumerate(meta):
    if m["year"] is not None and m["journal_id"]:
        journal_pairs[m["journal_id"]].append((int(m["year"]), i))

A_cent = build_year_centroids(author_pairs)
J_cent = build_year_centroids(journal_pairs)

def centroid_before(entry, year):
    """Zentroid, Anzahl, Cutoff-Jahr der Paper STRIKT vor `year`."""
    if entry is None:
        return None, 0, None
    ys, cs = entry
    i = bisect.bisect_left(ys, year)
    if i == 0:
        return None, 0, None
    return cs[i - 1] / i, i, ys[i - 1]

def count_before(entry, year):
    """Nur Anzahl - ohne den Zentroid zu bilden."""
    if entry is None:
        return 0
    return bisect.bisect_left(entry[0], year)

# =========================================================================
# 0) PRUEFUNG: Journalprofile duerfen nicht von der Autorenauswahl abhaengen
#
#    Verglichen wird gegen eine unabhaengige Zaehlung in der Datenbank, die
#    chosen_ids nirgends erwaehnt. Die DB kennt allerdings den Filter aus
#    Abschnitt 6 nicht: Paper, deren Abstract-Index zu leerem Text zerfaellt,
#    sind in der DB-Zaehlung enthalten, in meta aber nicht. Diese Differenz
#    ist bekannt und wird abgezogen - sie darf die Pruefung nicht ersetzen.
#
#    NICHT die Zaehlung aus meta selbst aufbauen. Das waere ein Vergleich von
#    meta mit meta, der immer besteht und nichts nachweist.
# =========================================================================
_db = defaultdict(dict)
for _j, _y, _n in con.execute(f"""
    SELECT primary_location.source.id AS journal_id,
           publication_year           AS year,
           COUNT(DISTINCT id)         AS n
    FROM {TBL}
    WHERE title IS NOT NULL AND abstract_inverted_index IS NOT NULL
      AND primary_location.source.id IS NOT NULL
      AND publication_year IS NOT NULL
    GROUP BY 1, 2
""").fetchall():
    _db[short_id(_j)][int(_y)] = _n

# bekannte Abweichung: leere Abstracts, protokolliert in Abschnitt 6
_drop = defaultdict(dict)
for _j, _years in dropped_empty_abstract.items():
    for _y, _n in _years.items():
        _drop[short_id(_j)][_y] = _n

def _db_before(j, y):
    """DB-Zaehlung strikt vor y, bereinigt um die leeren Abstracts."""
    total = sum(n for yy, n in _db.get(j, {}).items() if yy < y)
    lost  = sum(n for yy, n in _drop.get(j, {}).items() if yy < y)
    return total - lost

_bad = []
for _j in list(J_cent)[:200]:                 # Stichprobe von 200 Journalen
    for _y in range(2015, 2025):
        if count_before(J_cent.get(_j), _y) != _db_before(_j, _y):
            _bad.append((_j, _y,
                         count_before(J_cent.get(_j), _y),
                         _db_before(_j, _y)))
print(f"Journalprofil-Pruefung: {len(_bad)} Abweichungen gegen die DB "
      f"(erwartet: 0)   |   leere Abstracts abgezogen: {n_dropped}")
assert not _bad, (
    "Journalprofil stimmt nicht mit der unabhaengigen DB-Zaehlung ueberein. "
    "Erste Faelle (journal, jahr, aus_profil, aus_db): " + str(_bad[:3])
)

# =========================================================================
# 1) n_journal_papers fuer ALLE Zeilen - haengt nur am Journal-Jahr
# =========================================================================
uj = ev[[COL_JOURNAL, COL_YEAR]].drop_duplicates()
jc = {(j, int(y)): count_before(J_cent.get(j), int(y))
      for j, y in zip(uj[COL_JOURNAL].values, uj[COL_YEAR].values)}
print(f"{len(jc):,} eindeutige Journal-Jahr-Kombinationen.")

n_jrn_all = np.fromiter(
    (jc.get((j, int(y)), 0) for j, y in zip(ev[COL_JOURNAL].values,
                                            ev[COL_YEAR].values)),
    dtype=np.int64, count=len(ev))

# =========================================================================
# 2) topic_match nur dort, wo ein Autor-Profil existieren kann
# =========================================================================
embedded = set(author_pairs)
mask = ev[COL_AUTHOR].isin(embedded).values
sub  = ev.loc[mask]
print(f"{mask.sum():,} Zeilen mit eingebettetem Autor, "
      f"{(~mask).sum():,} ohne.")

ev_a = sub[COL_AUTHOR].values
ev_j = sub[COL_JOURNAL].values
ev_y = sub[COL_YEAR].astype(int).values

def unique_centroids(keys_years, cent):
    uniq = sorted(set(keys_years))
    ix   = {k: i for i, k in enumerate(uniq)}
    M    = np.full((len(uniq), X.shape[1]), np.nan)
    N    = np.zeros(len(uniq), dtype=np.int32)
    CUT  = np.full(len(uniq), -1, dtype=np.int32)
    for k in uniq:
        c, n, cut = centroid_before(cent.get(k[0]), k[1])
        if c is not None:
            M[ix[k]], N[ix[k]], CUT[ix[k]] = c, n, cut
    return ix, M, N, CUT

ay_ix, AY, AN, ACUT = unique_centroids(zip(ev_a.tolist(), ev_y.tolist()), A_cent)
jy_ix, JY, _,  _    = unique_centroids(zip(ev_j.tolist(), ev_y.tolist()), J_cent)
ia = np.array([ay_ix[(a, y)] for a, y in zip(ev_a, ev_y)])
ij = np.array([jy_ix[(j, y)] for j, y in zip(ev_j, ev_y)])

n_prof = AN[ia]
cutoff = ACUT[ia]
n_jrn  = n_jrn_all[mask]          # dieselbe Quelle wie oben - keine Abweichung

vals = np.full(len(sub), np.nan)
CH = 200_000
for s in range(0, len(sub), CH):
    e  = slice(s, s + CH)
    d2 = np.sum((AY[ia[e]] - JY[ij[e]]) ** 2, axis=1)
    vals[e] = np.exp(-0.5 * d2 / sig2)

has_a = n_prof > 0
has_j = n_jrn  > 0
status_sub = np.where(has_a & has_j, "ok",
             np.where(~has_a & ~has_j, "no_author_and_journal_history",
             np.where(~has_a, "no_author_history", "no_journal_history")))

# =========================================================================
# 3) Status fuer Autoren ohne Profil
# =========================================================================
cnt_usable = con.execute("""
    SELECT author_id, COUNT(DISTINCT work_id) AS n
    FROM author_paper GROUP BY author_id
""").fetchall()
n_usable = {short_id(a): n for a, n in cnt_usable}

cnt_total = con.execute(f"""
    WITH exploded AS (
        SELECT id AS work_id, unnest(authorships) AS a FROM {TBL}
    )
    SELECT a.author.id AS author_id, COUNT(DISTINCT work_id) AS n
    FROM exploded WHERE a.author.id IS NOT NULL
    GROUP BY a.author.id
""").fetchall()
n_total = {short_id(a): n for a, n in cnt_total}

nonemb_a = ev.loc[~mask, COL_AUTHOR].values
tot = np.array([n_total.get(a, 0) for a in nonemb_a])
# v5: bei MIN_PAPERS = 1 entfaellt "below_threshold_unproductive" fast
# vollstaendig. Uebrig bleiben Autoren ohne Werk in der DB und Autoren,
# deren Werke keinen Abstract haben - also nicht einbettbar sind.
status_non = np.where(tot == 0, "author_not_in_db",
             np.where(tot < MIN_PAPERS, "below_threshold_unproductive",
                                        "below_threshold_no_abstract"))

# =========================================================================
# 4) Schreiben
# =========================================================================
ev[COL_TOPIC]          = np.nan
ev["n_journal_papers"] = n_jrn_all          # fuer alle Zeilen
ev["n_profile_papers"] = pd.NA              # Default: nicht berechnet
ev["profile_cutoff"]   = pd.NA
ev["tm_status"]        = ""

ev.loc[mask, COL_TOPIC]          = vals
ev.loc[mask, "n_profile_papers"] = n_prof
ev.loc[mask, "profile_cutoff"]   = np.where(cutoff >= 0, cutoff, np.nan)
ev.loc[mask, "tm_status"]        = status_sub
ev.loc[~mask, "tm_status"]       = status_non

for c in ["n_profile_papers", "n_journal_papers", "profile_cutoff"]:
    ev[c] = ev[c].astype("Int64")

# =========================================================================
# 5) Report + Konsistenzpruefung
# =========================================================================
n = int(ev[COL_TOPIC].notna().sum())
print(f"\ngefuellt: {n:,} von {len(ev):,} ({n/len(ev):.1%})")
if n:
    v = ev[COL_TOPIC].values
    print(f"topic_match: Mittel {np.nanmean(v):.3f}, Median {np.nanmedian(v):.3f}")

print("\n=== tm_status ===")
print(ev["tm_status"].value_counts().to_string())

# Muss 0 sein: n_journal_papers darf pro Journal-Jahr nur EINEN Wert haben
nuniq = ev.groupby([COL_JOURNAL, COL_YEAR])["n_journal_papers"].nunique()
print(f"\nKonsistenz n_journal_papers: {(nuniq > 1).sum()} Journal-Jahre "
      f"mit mehr als einem Wert (erwartet: 0)")

# Muss 0 sein: ein leeres topic_match darf nie als 0 im Export landen
_zero = int(((ev[COL_TOPIC] == 0) & (ev["tm_status"] != "ok")).sum())
print(f"Leere topic_match-Werte als 0 kodiert: {_zero} (erwartet: 0)")
assert _zero == 0

# Muss 0 sein: profile_cutoff liegt immer strikt vor t
_cut = int((ev["profile_cutoff"].notna() & (ev["profile_cutoff"] >= ev[COL_YEAR])).sum())
print(f"profile_cutoff >= t: {_cut} Zeilen (erwartet: 0)")
assert _cut == 0

# Zeilenzahl darf sich durch das Fuellen nicht veraendert haben
print(f"Zeilen: {len(ev):,} (Eingang unveraendert)")

print(f"\nMIN_PAPERS = {MIN_PAPERS} | sig2 = {sig2:.4f}")
print("Dieser Lauf ist mit einem anderen nur vergleichbar, wenn BEIDE Werte "
      "uebereinstimmen. topic_match-Werte aus verschiedenen Laeufen niemals mischen.")


In [ ]:
# Zurueckschreiben - topic_match + Diagnosespalten, alle anderen unveraendert
# Int64-Spalten werden von pandas korrekt als "2020" (nicht "2020.0") geschrieben.
ev.to_csv(EVENT_OUT, index=False)
print(f"Geschrieben: {EVENT_OUT}")
print(f"{len(ev):,} Zeilen, {len(ev.columns)} Spalten.")
print("\nNeue Spalten: n_profile_papers, n_journal_papers, profile_cutoff, tm_status")
print("Leeres topic_match ist NIE 0 - der Grund steht in tm_status.")
print("\ndtypes:\n", ev.dtypes.to_string())


## 15 - Ergebnisse zurück nach SharePoint

Erst nach erfolgreichem Lauf ausführen. Wir laden die drei CSVs und eine kurze
Dateiliste mit Eingabestand, Code-Stand und Paketversionen in einen neuen Laufordner.
Anschließend lesen wir jede Datei zurück und prüfen ihren Hash. Dieser neue Lauf
ersetzt nicht den geprüften v5-Release; dafür müssen die Ergebnisse erst verglichen werden.

In [ ]:
import hashlib, importlib.metadata, json
outputs = (LOCAL_CSV, LOCAL_KEYS, EVENT_OUT)
if not all(p.is_file() for p in outputs):
    raise RuntimeError('Erst die drei CSV-Exporte vollständig erzeugen.')
def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not available'

receipt = {
    'run': RUN_ID, 'input_release': manifest['release'],
    'input_manifest_sha256': file_sha(REPO / 'data-manifest.json'),
    'repository_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip(),
    'notebook_source': 'Current Colab session; may contain unsaved edits',
    'rclone_version': RCLONE_VERSION,
    'versions': {p: package_version(p) for p in
                 ('duckdb', 'transformers', 'adapters', 'torch', 'pandas', 'numpy')},
    'files': [{'path': str(p.relative_to(RUN_ROOT)), 'bytes': p.stat().st_size,
               'sha256': file_sha(p)} for p in outputs]
}
receipt_path = RUN_ROOT / 'run.json'
receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
run_remote = REMOTE + '/runs/' + RUN_ID
for source in (*outputs, receipt_path):
    destination = run_remote + '/' + source.relative_to(RUN_ROOT).as_posix()
    transfer(source, destination)
    with tempfile.TemporaryDirectory() as staging:
        returned = Path(staging) / 'readback'
        transfer(destination, returned)
        if file_sha(source) != file_sha(returned):
            raise RuntimeError('Rückprüfung fehlgeschlagen: ' + source.name)
    print('Auf SharePoint gespeichert und zurückgeprüft:', source.relative_to(RUN_ROOT))
print('Laufordner:', run_remote)